# LARRY fixed point 1 gene-gradient analysis

Current FlowMap package version of the original FP1 gradient-field notebook. The old legacy fixed-point, LAP, and gene-gradient calls are replaced by `flowmap.geometry`.

In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd()
FIG_DIR = NOTEBOOK_DIR / "figures" / "larry"
FIG_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("NUMBA_CACHE_DIR", str((NOTEBOOK_DIR / ".numba_cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((NOTEBOOK_DIR / ".mpl_cache").resolve()))

LOCAL_FLOWMAP_SRC = Path("/Users/jh/Projects/flowmap/src")
if LOCAL_FLOWMAP_SRC.exists() and str(LOCAL_FLOWMAP_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_FLOWMAP_SRC))

DATA_PATH = (NOTEBOOK_DIR / "../data/flowmap_manuscript/larry/tutorial_larry_flowmap_data.joblib").resolve()
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Put the tutorial_larry_flowmap_data.joblib source data file there before running."
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

from flowmap.geometry import FixedPointAnalyzer, GeneGradientAnalyzer, LagrangianPathOptimizer
from flowmap.plot import plot_velocity_stream
from flowmap.utils import load_dataset

data = load_dataset(DATA_PATH)
emb = data.embedder
labels = data.obs["state_info"].astype(str).to_numpy()

X_gene = data.layers["spliced"]
V_gene = data.layers["velocity"]
emb.n_spline_points = None
emb.gene_names = np.asarray(data.var_names)

print(data)
print("Embedding:", emb.X_emb.shape, "Velocity:", emb.V_emb.shape)

In [ ]:
fpa = FixedPointAnalyzer(emb)
fixed_points = fpa.identify_fixed_points(
    grid_resolution=90,
    speed_smoothing=1.2,
    speed_quantile_threshold=0.06,
)

fp_table = pd.DataFrame([
    {
        "fixed_point": i,
        "x": fp["position"][0],
        "y": fp["position"][1],
        "type": fp.get("type", "unknown"),
        "speed": fp.get("speed", np.nan),
    }
    for i, fp in enumerate(fixed_points)
])
fp_table

In [ ]:
def plot_stream_with_fixed_points(ax, title=None):
    plot_velocity_stream(
        emb.X_emb,
        spline=emb.spline_vf,
        scatter_color=labels,
        scatter_size=8,
        scatter_alpha=0.45,
        ax=ax,
    )
    for i, fp in enumerate(fixed_points):
        pos = fp["position"]
        ax.scatter(pos[0], pos[1], c="red", marker="x", s=220, linewidths=3, zorder=20)
        ax.text(pos[0] + 0.15, pos[1] + 0.15, str(i + 1), color="red", fontsize=14, weight="bold")
    if title:
        ax.set_title(title)
    ax.set_aspect("equal")
    return ax


def fit_paths(root, endpoints, names):
    lap = LagrangianPathOptimizer(emb, D=1.0, lam=1e-4)
    rows = []
    for name, end in zip(names, endpoints):
        result = lap.fit_path(start=np.asarray(root), end=np.asarray(end), distance_mode="orig")
        rows.append({"name": name, "end": np.asarray(end), "result": result, "path": result["path_refined"]})
    return rows


def cells_near_path(path, radius_fraction=0.05):
    scale = np.mean(np.ptp(emb.X_emb, axis=0))
    radius = radius_fraction * scale
    distances = np.linalg.norm(emb.X_emb[:, None, :] - path[None, :, :], axis=2).min(axis=1)
    return np.where(distances <= radius)[0], radius


def plot_paths(paths, root, title, filename):
    fig, ax = plt.subplots(figsize=(7, 7))
    plot_stream_with_fixed_points(ax, title=title)
    ax.scatter(root[0], root[1], c="limegreen", s=90, edgecolor="black", zorder=30, label="start")
    for row in paths:
        path = row["path"]
        ax.plot(path[:, 0], path[:, 1], lw=3, label=row["name"], zorder=25)
        ax.scatter(path[-1, 0], path[-1, 1], c="black", s=45, zorder=30)
    ax.legend(frameon=False, fontsize=9, loc="best")
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    return fig, ax


def ensure_gene_splines():
    if not hasattr(emb, "tps_gene") or emb.tps_gene is None:
        emb.fit_gene_level_splines(X=X_gene, V=V_gene, dof_gene=50, dof_vf_gene=50)


def summarize_gene_gradients(path_rows):
    ensure_gene_splines()
    summaries = []
    for row in path_rows:
        neighbor_indices, radius = cells_near_path(row["path"])
        analyzer = GeneGradientAnalyzer(emb, cell_indices=neighbor_indices, verbose=True)
        out = analyzer.compute_relative_gradients(
            neighbor_indices,
            np.arange(len(emb.gene_names)),
            weight="magnitude",
        )
        df = pd.DataFrame({
            "path": row["name"],
            "gene_idx": out["gene_indices"],
            "gene": emb.gene_names[out["gene_indices"]],
            "relative_angle": out["angles"],
            "magnitude": out["magnitudes"],
            "n_path_cells": len(neighbor_indices),
            "path_radius": radius,
        }).sort_values("magnitude", ascending=False)
        summaries.append({"path": row["name"], "neighbors": neighbor_indices, "result": out, "df": df})
    return summaries


def plot_gradient_polar(summary, filename, title=None, highlight_genes=None):
    df = summary["df"].copy()
    theta = np.deg2rad(df["relative_angle"].to_numpy())
    r = df["magnitude"].to_numpy()
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="polar")
    ax.scatter(theta, r, s=16, c=df["relative_angle"], cmap="coolwarm", alpha=0.65)
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(-1)
    ax.set_title(title or summary["path"])
    if highlight_genes:
        sub = df[df["gene"].isin(highlight_genes)]
    else:
        sub = df.head(8)
    for _, row in sub.iterrows():
        ax.text(np.deg2rad(row["relative_angle"]), row["magnitude"], row["gene"], fontsize=9)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    return fig, ax


def plot_path_gene_heatmap(summary, path_row, filename, top_n=15):
    df = summary["df"].copy()
    forward = df[np.abs(df["relative_angle"]) <= 30].head(top_n)
    if forward.empty:
        forward = df.head(top_n)
    gene_indices = forward["gene_idx"].to_numpy()
    genes = forward["gene"].to_numpy()
    path = path_row["path"]
    expr = emb.tps_gene.predict(path)[:, gene_indices]
    expr = np.asarray(expr)
    expr = gaussian_filter1d(expr, sigma=1, axis=0)
    expr = (expr - expr.mean(axis=0, keepdims=True)) / (expr.std(axis=0, keepdims=True) + 1e-9)
    fig, ax = plt.subplots(figsize=(7, max(4, 0.3 * len(genes))))
    im = ax.imshow(expr.T, aspect="auto", cmap="vlag" if "vlag" in plt.colormaps() else "coolwarm", vmin=-2, vmax=2)
    ax.set_yticks(np.arange(len(genes)))
    ax.set_yticklabels(genes)
    ax.set_xlabel("Path position")
    ax.set_title(path_row["name"])
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    return fig, ax


def get_gene_vector(matrix, idx):
    col = matrix[:, idx]
    if hasattr(col, "toarray"):
        return np.asarray(col.toarray()).ravel()
    return np.asarray(col).ravel()

def plot_gene_expression_strip(genes, filename, title_prefix=""):
    gene_names = np.asarray(emb.gene_names)
    valid = [g for g in genes if g in gene_names]
    if not valid:
        raise ValueError(f"None of these genes were found: {genes}")
    fig, axes = plt.subplots(1, len(valid), figsize=(4 * len(valid), 4), squeeze=False)
    for ax, gene in zip(axes.ravel(), valid):
        idx = int(np.where(gene_names == gene)[0][0])
        vals = get_gene_vector(X_gene, idx)
        ax.scatter(emb.X_emb[:, 0], emb.X_emb[:, 1], c=vals, s=5, cmap="magma_r", alpha=0.65, linewidths=0)
        ax.set_title(f"{title_prefix}{gene}")
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    return fig, axes

In [ ]:
fp1 = fixed_points[0]["position"]
fp2 = fixed_points[1]["position"]

endpoints = [
    np.array([-6.0, 6.0]),
    np.array([2.3, -7.6]),
    fp2,
]
path_names = ["fp1_to_upper_left", "fp1_to_terminal", "fp1_to_fp2"]

path_rows = fit_paths(fp1, endpoints, path_names)
plot_paths(path_rows, fp1, "Fixed point 1 paths", "fp1_paths_stream.png")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_stream_with_fixed_points(ax, title="Fixed point 1 local path cells")
for row in path_rows:
    neighbor_indices, radius = cells_near_path(row["path"])
    ax.scatter(emb.X_emb[neighbor_indices, 0], emb.X_emb[neighbor_indices, 1], s=10, alpha=0.6, label=row["name"])
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fp1_path_neighbor_cells.png", dpi=300, bbox_inches="tight")

In [ ]:
gradient_summaries = summarize_gene_gradients(path_rows)
all_gradients = pd.concat([s["df"] for s in gradient_summaries], ignore_index=True)
all_gradients.to_csv(FIG_DIR / "fp1_gene_gradient_summary.csv", index=False)

for summary, path_row in zip(gradient_summaries, path_rows):
    safe_name = summary["path"].replace(" ", "_").replace("→", "to").replace("-", "_").lower()
    plot_gradient_polar(
        summary,
        f"fp1_{safe_name}_polar_gene_gradients.png",
        title=summary["path"],
        highlight_genes=["Mmp12","Rrm2","H2-Aa","BC100530"],
    )
    plot_path_gene_heatmap(
        summary,
        path_row,
        f"fp1_{safe_name}_path_gene_heatmap.png",
        top_n=15,
    )

all_gradients.head(20)

In [ ]:
plot_gene_expression_strip(
    ["Mmp12", "Rrm2", "H2-Aa", "BC100530"],
    "fp1_highlight_gene_expression.png",
)